# Paper figures

The figures and the table of the paper from the CSV files in `results/` (saved as PNG in `results/figures/paper/`):

1. **Offline label shift** (calibration with a class mix at $\mathrm{KL}(q_{\text{cal}} \Vert q_{\text{test}})$, the whole test split): coverage and average class set size of the 5 methods vs KL; 2 × 5 grid (average, then each dataset).
2. **Online martingales**: mean $\log_{10} M_t$ over the streams for device shift L1 and L3 (blur, noise), domain shift and label shift (one trace per KL), unweighted and weighted ($w_{online}$).
3. **Tables** (α = 1 %): alarm % after the change, false alarm % and mean delay per shift case; per dataset: alarm rate after the change and the sizes of the calibration set and of the stream before / after the change.
4. **Online sets under label shift (KL = 0.5)**: moving average of the last 50 images of the coverage and the average class set size, FCP vs FCP + $w_{online}$; 2 × 5 grid (average, then each dataset).

Online streams: the images after the change keep their own class frequencies (test split, external set or corrupted test split, up to 600); the calibration set (16·C) and the first images of the stream (up to 200) are train images with a mix $q$: their frequencies, or a Dirichlet draw at the KL (label shift).


In [1]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Nimbus Sans"]
plt.rcParams['font.weight'] = 'bold'
import seaborn as sns
import os
# configs
sns.set_theme(style="ticks")

sys.path.insert(0, os.path.abspath('../'))  # project root
from utils.embedding_datasets import load_dataset

CSV_DIR, FIG_DIR = 'results', 'results/figures/paper'
os.makedirs(FIG_DIR, exist_ok=True)
CP_ALPHA, ALPHA, MA_WINDOW, MIN_STREAMS, PRE, POST = 0.10, 0.01, 50, 0.5, 200, 600
DATASETS = ['CheXpert5x200', 'nihlt', 'covid', 'NCT', 'SICAPv2', 'Skin', 'MESSIDOR', 'FIVES', 'MMAC']
KLS = [0.1, 0.25, 0.5, 0.8]
LABEL = [f'label shift (KL = {kl:g})' for kl in KLS]
MA_SHIFT = 'label shift (KL = 0.5)'
N_CLASSES = {n: load_dataset(n, os.path.abspath('../local_data/embeddings'))['n_classes'] for n in DATASETS}   # set size (%) = class set size / C · 100, per dataset
# colour-blind safe (Okabe-Ito) + line styles and markers
STYLE = {'ICP': ('#0072B2', '-', 'o'), 'FCP+$w_{oracle}$': ('#000000', '--', 'v'), 'FCP': ('#D55E00', '-', 's'),
         'FCP+mondrian': ('#009E73', '-.', 'D'), 'FCP+$w_{label}$': ('#E69F00', ':', '^')}
W_SHOW = {'unweighted': 'unweighted', 'w_online': 'weighted ($w_{online}$)'}
SET_STYLE = {'fcp': ('FCP', '#0072B2', '-'), 'fcp_w': ('FCP + $w_{online}$', '#D55E00', '--'), 'fcp_o': ('FCP + $w_{oracle}$', '#000000', '--')}
KL_STYLE = dict(zip(LABEL, [('#56B4E9', ':', 'o'), ('#0072B2', '--', 's'), ('#009E73', '-.', '^'), ('#E69F00', '-', 'D'), ('#D55E00', '-', 'v')]))

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, f'{name}.png'), dpi=300, bbox_inches='tight')

def style_ax(ax, title, bold=False):
    ax.set_title(title, loc='left', fontsize=11, fontweight='bold' if bold else None)
    ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)

cut = lambda q: q[q['streams'] >= MIN_STREAMS * q['streams'].max()]      # online curves: stop below 50 % of the streams

# 1 · Offline label shift: the 5 methods vs KL

In [2]:
METHODS = ['FCP+mondrian', 'FCP+$w_{oracle}$', 'FCP+$w_{label}$','ICP', 'FCP']
LEGENDS = {'ICP': 'ICP', 'FCP': 'FCP', 'FCP+mondrian': 'FCP$_{CLASSWISE}$', 'FCP+$w_{oracle}$': 'FCP+$w_{oracle}$', 'FCP+$w_{label}$': 'FCP+$w_{ours}$'}

DIRS = {"coverage":"results/figures/paper/offline/cov/", "class_set_size":"results/figures/paper/offline/set/"}
os.makedirs(DIRS["coverage"], exist_ok=True)
os.makedirs(DIRS["class_set_size"], exist_ok=True)

r13 = pd.read_csv(os.path.join(CSV_DIR, 'offline_shifts_testmix_100draws.csv'))
r13 = r13[r13['shift'].str.startswith('label shift')].assign(target_KL=lambda t: t['shift'].str.extract(r'= ([\d.]+)')[0].astype(float))
r13['class_set_size'] = 100 * r13['class_set_size'] / r13['dataset'].map(N_CLASSES)     # set size (%) per dataset, then averaged
per_ds = r13.groupby(['dataset', 'method', 'target_KL'])[['coverage', 'class_set_size']].mean().reset_index()
curves = pd.concat([per_ds.groupby(['method', 'target_KL'])[['coverage', 'class_set_size']].mean().reset_index().assign(dataset='average'), per_ds])
for col, ylabel in [('coverage', 'Coverage (%)'), ('class_set_size', 'Set size (%)')]:
  for ds in ['average']:
    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    q = curves[curves['dataset'] == ds]
    for m in METHODS:
      qm = q[q['method'] == m].sort_values('target_KL'); c, ls, mk = STYLE[m]
      if col == 'class_set_size':
       ax.plot(qm['target_KL'], qm[col], color=c, ls=ls, marker=mk, ms=7, lw=2.5, label=m)
      else:
       ax.plot(qm['target_KL'], qm[col], color=c, ls=ls, marker=mk, ms=7, lw=2.5, label=m)
    if col == 'coverage': ax.axhline(100 * (1 - CP_ALPHA), color='gray', ls=':', lw=2)
    ks = sorted(q['target_KL'].unique())
    ax.set_xscale('log'); ax.minorticks_off(); ax.set_xticks(ks, [f'{k:g}' for k in ks]); ax.set_xlabel('KL')
    #style_ax(ax, ds, ds == 'average')

    plt.xlabel(r"KL($q_{\text{cal}}||q_{\text{test}}$) - log scaled", fontsize=15, weight="bold")
    plt.ylabel(ylabel, fontsize=15, weight="bold")
    plt.tick_params(axis='both', which='major', labelsize=14)
    ax = plt.gca()
    plt.grid(axis="both", linestyle='--', linewidth=2, zorder=-1)
    ax = plt.gca()
    for tick in ax.get_xticklabels():
            tick.set_fontweight('bold')
    for tick in ax.get_yticklabels():
            tick.set_fontweight('bold')
    plt.tight_layout()

    if ds == "average":
      if col == "coverage":
        plt.legend(labels=[LEGENDS[method] for method in METHODS], loc='lower left', prop={'weight': 'bold', 'size': 14}, framealpha=1, ncols=2, handletextpad=0.6, columnspacing=0.4, handlelength=1.0)
        plt.yticks([84.0, 86.0, 88.0, 90.0, 92.0, 94.0, 96.0], [84, 86, 88, 90, 92, 94, 96])
      if col == "class_set_size":
        #plt.legend(labels=[LEGENDS[method] for method in METHODS], loc='upper left', prop={'weight': 'bold', 'size': 14}, framealpha=1, ncols=2, handletextpad=0.6, columnspacing=0.4, handlelength=1.0)
        plt.yticks([35, 40, 45, 50, 55, 60, 65, 70, 75], [35, 40, 45, 50, 55, 60, 65, 70, 75])

    path = DIRS[col] + ds + ".png"
    plt.savefig(path, dpi=200, format='png', bbox_inches='tight')
    plt.close()


findfont: Failed to find font weight bold for STIXSizeFiveSym, now using 400.
findfont: Failed to find font weight bold for cmmi10, now using 400.
findfont: Failed to find font weight bold for cmsy10, now using 400.
findfont: Failed to find font weight bold for cmr10, now using 400.
findfont: Failed to find font weight bold for cmtt10, now using 400.
findfont: Failed to find font weight bold for cmti10, now using 400.
findfont: Failed to find font weight bold for cmb10, now using 400.
findfont: Failed to find font weight bold for cmss10, now using 400.
findfont: Failed to find font weight bold for cmex10, now using 400.
findfont: Failed to find font weight bold for DejaVu Sans Display, now using 400.


# 2 · Online martingales: device (L1, L3), domain and label shift

In [3]:
marting_dir = "results/figures/paper/martingale/"
os.makedirs(marting_dir, exist_ok=True)
w = 'unweighted'
mart = pd.read_csv(os.path.join(CSV_DIR, 'online_martingale_curves.csv'))
LABEL_T_MAX = 200                     # label shift panels: up to image 300 after the change
panels = [
          ('label shift', {s: KL_STYLE[s][:2] for s in LABEL}),
          ('device shift L1', {'blur L1': ('#0072B2', '-'), 'noise L1': ('#E69F00', '--')}),
          ('device shift L3', {'blur L3': ('#0072B2', '-'), 'noise L3': ('#E69F00', '--')}),
          ('domain shift', {'domain shift': ('#000000', '-')}),
          ]
legends = {'label shift': ['$D_{KL}=0.10$','$D_{KL}=0.25$','$D_{KL}=0.50$','$D_{KL}=0.80$'],
           'device shift L1': ['Blur L1', 'Noise L1'],
           'device shift L3': ['Blur L3', 'Noise L3']}
for (title, traces) in panels:
  fig, ax = plt.subplots(1, 1, figsize=(5, 3))
  for s, (c, ls) in traces.items():
    q = cut(mart[(mart['weighting'] == w) & (mart['shift'] == s)])
    q = q[q['t'] <= LABEL_T_MAX]
    ax.plot(q['t'], q['mean'], color=c, ls=ls, lw=2.5, label=s.replace('label shift ', ''))

  if title != "domain shift":
    plt.legend(labels=legends[title], loc='upper left', prop={'weight': 'bold', 'size': 14},
              framealpha=1, ncols=1, handletextpad=0.6, columnspacing=0.4, handlelength=1.0)

  for s, (c, ls) in traces.items():
    q = cut(mart[(mart['weighting'] == w) & (mart['shift'] == s)])
    q = q[q['t'] <= LABEL_T_MAX]
    ax.fill_between(q['t'], q['q25'], q['q75'], color=c, alpha=0.10, lw=0)

  ax.axvline(0, color='black', ls=':', lw=2); ax.axhline(np.log10(1 / ALPHA), color='gray', ls=':', lw=2)
  ax.set_xlim(-PRE - 10, LABEL_T_MAX)
  ax.set_ylim(-1, 20)

  plt.xlabel("Images relative to shift", fontsize=15, weight="bold")
  plt.ylabel('log$_{10}(M_t)$', fontsize=15, weight="bold")
  plt.tick_params(axis='both', which='major', labelsize=14)
  ax = plt.gca()
  plt.grid(axis="both", linestyle='--', linewidth=2, zorder=-1, alpha=0.5)
  ax = plt.gca()
  for tick in ax.get_xticklabels():
    tick.set_fontweight('bold')
  for tick in ax.get_yticklabels():
    tick.set_fontweight('bold')
  plt.tight_layout()

  if title=='domain shift':
    plt.text(-180, 15.5, s="Natural shift",fontsize=14,
        fontweight='bold',
        bbox=dict(
            facecolor='white',
            edgecolor='black',
            alpha=0.8,
            boxstyle='round,pad=0.3'))
  if title=='label shift':
    plt.text(25, 15.5, s="Label shift",fontsize=14,
        fontweight='bold',
        bbox=dict(
            facecolor='white',
            edgecolor='black',
            alpha=0.8,
            boxstyle='round,pad=0.3'))
  if "device shift" in title:
    plt.text(25, 15.5, s="Device shift",fontsize=14,
    fontweight='bold',
    bbox=dict(
        facecolor='white',
        edgecolor='black',
        alpha=0.8,
        boxstyle='round,pad=0.3'))

  plt.yticks([2, 6, 10, 14, 18])
  plt.xticks([-200, -150, -100, -50, 0, 50, 100, 150, 200], labels=[-200, "", -100, "", 0, "", 100, "", 200])

  path = marting_dir + title + ".png"
  plt.savefig(path, dpi=200, format='png', bbox_inches='tight')
  plt.close()



# 4 · Online sets under label shift (KL = 0.5): moving average of the last 50 images

In [4]:
"""
Metrics of conformal prediction sets (boolean (N, C) matrices, sets[i, c] = class c in the set of image i).

- set_metrics:          coverage (%), average class set size, set size, UCV (pp), CCV (pp), % all-class sets
- moving_window_counts: per-class counts of the last `window` images at every image of a stream
- pooled_metrics:       the metrics of set_metrics from counts pooled over streams (moving averages)
"""

import numpy as np


def _class_coverages(pred_sets, labels, C):
    cov = np.full(C, np.nan)
    for c in range(C):
        if (labels == c).any():
            cov[c] = pred_sets[labels == c, c].mean()
    return cov[np.isfinite(cov)]


def set_metrics(pred_sets, labels, C, alpha=0.1):
    """coverage (%), class_set_size (mean over the classes present of their mean set size), set_size,
    UCV = mean max(0, (1 - alpha) - cov_c) (pp), CCV = mean |cov_c - (1 - alpha)| (pp), all_class (% of all-class sets)."""
    labels = np.asarray(labels).astype(int)
    if len(labels) == 0:
        return {k: float('nan') for k in ['coverage', 'class_set_size', 'set_size', 'UCV', 'CCV', 'all_class']}
    cov, size = _class_coverages(pred_sets, labels, C), pred_sets.sum(1)
    return {'coverage': 100 * float(pred_sets[np.arange(len(labels)), labels].mean()),
            'class_set_size': float(np.mean([size[labels == c].mean() for c in range(C) if (labels == c).any()])),
            'set_size': float(size.mean()),
            'UCV': 100 * float(np.mean(np.maximum(0, (1 - alpha) - cov))),
            'CCV': 100 * float(np.mean(np.abs(cov - (1 - alpha)))),
            'all_class': 100 * float(pred_sets.all(1).mean())}


def moving_window_counts(labels, covered, size, C, window=50):
    """Counts of the last `window` images (fewer at the start) at every image t of a stream: n_c, covered_c, size_c
    (M, C) per class and n_all_class (M,). Sum them over streams, then pooled_metrics at every t."""
    labels = np.asarray(labels).astype(int); M = len(labels)
    Y = np.zeros((M, C)); Y[np.arange(M), labels] = 1
    def win(X):
        cs = np.cumsum(X, 0)
        return cs - np.concatenate([np.zeros((window,) + X.shape[1:]), cs])[:M]
    return (win(Y), win(Y * np.asarray(covered, float)[:, None]), win(Y * np.asarray(size, float)[:, None]),
            win((np.asarray(size) == C).astype(float)))


def pooled_metrics(n_c, covered_c, size_c, n_all_class, alpha=0.1):
    """set_metrics (without set_size) from per-class counts pooled over streams, plus n (images)."""
    n_c, covered_c, size_c = (np.asarray(a, float) for a in (n_c, covered_c, size_c))
    pres = n_c > 0
    cov, n = covered_c[pres] / n_c[pres], n_c.sum()
    return {'n': int(n), 'coverage': 100 * covered_c.sum() / n, 'class_set_size': float(np.mean(size_c[pres] / n_c[pres])),
            'UCV': 100 * float(np.mean(np.maximum(0, 1 - alpha - cov))), 'CCV': 100 * float(np.mean(np.abs(cov - (1 - alpha)))),
            'all_class': 100 * n_all_class / n}

In [5]:
CP_ALPHA, ALPHA, MA_WINDOW, MIN_STREAMS, PRE, POST = 0.10, 0.01, 50, 0.5, 200, 600
online_cov_dir = "results/figures/paper/online_cov/"
os.makedirs(online_cov_dir, exist_ok=True)

cols = ['case', 'shift', 'rep', 'C', 't', 'y', 'covered_fcp', 'covered_fcp_w', 'covered_fcp_o', 'size_fcp', 'size_fcp_w', 'size_fcp_o']
st = pd.concat(ch[ch['shift'] == MA_SHIFT] for ch in pd.read_csv(os.path.join(CSV_DIR, 'online_streams.csv'), usecols=cols, chunksize=2_000_000))
T_AX = np.arange(-PRE, POST)
acc = {}
for (case, rep), g in st.groupby(['case', 'rep'], sort=False):
    t = g['t'].to_numpy(); k = (t >= -PRE) & (t < POST); C = int(g['C'].iat[0]); y = g['y'].to_numpy()
    for m in SET_STYLE:
        n, cov, size, n_all = moving_window_counts(y, g[f'covered_{m}'].to_numpy(), g[f'size_{m}'].to_numpy(), C, MA_WINDOW)
        for grp in ['average', case]:
            a = acc.setdefault((grp, m), [np.zeros((len(T_AX), 64)) for _ in range(3)] + [np.zeros(len(T_AX)), np.zeros(len(T_AX))])
            idx = t[k] + PRE
            a[0][idx, :C] += n[k]; a[1][idx, :C] += cov[k]; a[2][idx, :C] += size[k]; a[3][idx] += n_all[k]; a[4][idx] += 1
ma = pd.DataFrame([{'group': grp, 'method': m, 't': int(T_AX[j]), 'streams': int(a[4][j]),
                    **pooled_metrics(a[0][j], a[1][j], a[2][j], a[3][j], CP_ALPHA)}
                   for (grp, m), a in acc.items() for j in range(len(T_AX)) if a[0][j].sum() > 0])
# set size (%): class set size / C · 100 per dataset; average = mean of the per-dataset curves, only where every dataset has its curve
ma['set_size_pct'] = 100 * ma['class_set_size'] / ma['group'].map(N_CLASSES)
per = pd.concat([cut(g) for _, g in ma[ma['group'] != 'average'].groupby(['group', 'method'])])
avg = per.groupby(['method', 't']).agg(set_size_pct=('set_size_pct', 'mean'), streams=('streams', 'sum'), datasets=('group', 'size')).reset_index()
ma.loc[ma['group'] == 'average', 'set_size_pct'] = np.nan
ma = pd.concat([ma, avg[avg['datasets'] == per['group'].nunique()].drop(columns='datasets').assign(group='average')], ignore_index=True)
for col, ylabel in [('coverage', 'Coverage (%)'), ('set_size_pct', 'Set size (%)')]:
    fig, ax = plt.subplots(1, 1, figsize=(5, 3))
    for grp in ['average']:
        q = ma[ma['group'] == grp]
        if q.empty:
            ax.set_visible(False); continue
        for m, (lab, c, ls) in SET_STYLE.items():
            qm = cut(q[(q['method'] == m) & q[col].notna()]).sort_values('t')
            #print(qm)
            ax.plot(qm['t'].iloc[MA_WINDOW:], qm[col].iloc[MA_WINDOW:], color=c, ls=ls, lw=1.8, label=lab)
        ax.axvline(0, color='black', ls=':', lw=2)
        if col == 'coverage': ax.axhline(100 * (1 - CP_ALPHA), color='gray', ls=':', lw=2)
        #style_ax(ax, grp, grp == 'average')

    plt.xlabel("Images relative to shift", fontsize=15, weight="bold")
    plt.ylabel(ylabel, fontsize=15, weight="bold")
    plt.tick_params(axis='both', which='major', labelsize=14)
    ax = plt.gca()
    plt.grid(axis="both", linestyle='--', linewidth=2, zorder=-1, alpha=0.5)
    ax = plt.gca()
    for tick in ax.get_xticklabels():
      tick.set_fontweight('bold')
    for tick in ax.get_yticklabels():
      tick.set_fontweight('bold')
    plt.tight_layout()

    plt.xticks([-200, -150, -100, -50, 0, 50, 100, 150, 200], labels=[-200, "", -100, "", 0, "", 100, "", 200])

    ax.set_xlim(-PRE - 10, LABEL_T_MAX)

    if col == "coverage":
      ax.set_ylim(76, 100)
      plt.legend(labels=["FCP", "FCP+$w_{ours}$", "FCP+$w_{oracle}$"], loc='lower left', prop={'weight': 'bold', 'size': 14},
            framealpha=1, ncols=1, handletextpad=0.6, columnspacing=0.4, handlelength=1.0)
      plt.yticks([76, 80, 84, 88, 92, 96, 100])
    # set size (%): automatic y limits (the oracle sets are much larger)

    path = online_cov_dir + col + ".png"
    plt.savefig(path, dpi=200, format='png', bbox_inches='tight')
    plt.close()


In [6]:
print(st['size_fcp'] / st['C'])

1618263    0.6
1618264    0.4
1618265    0.8
1618266    1.0
1618267    0.8
          ... 
2153370    0.4
2153371    0.4
2153372    0.4
2153373    0.8
2153374    0.4
Length: 535112, dtype: float64
